# 01 · Limpieza de datos — Proyecto VTC (VTC)

**Objetivo de este cuaderno:** pasar de los datos «en bruto» (lo que leyó el OCR en las fotos del móvil) a tablas **limpias, sin duplicados, con los tipos correctos y validadas** contra los totales que da la propia app.

> **¿Qué es limpiar datos?** Es preparar los datos para que se puedan analizar sin errores: quitar repetidos, corregir valores mal leídos, poner a cada columna su tipo correcto (fecha, número, texto), decidir qué hacer con los huecos y marcar los valores raros. Se dice que en un proyecto real el 60–80 % del tiempo se va en esto.

**Regla de oro:** nunca se modifica el original. Leemos copias y guardamos el resultado en `datos/limpios/`. Cada cambio queda apuntado en `datos/limpios/registro_cambios.csv`.

Pasos:
1. Cargar los viajes candidatos del OCR y mirarlos.
2. Aplicar las correcciones manuales (documentadas).
3. Quitar duplicados (por el solape entre fotos).
4. Añadir el método de pago (efectivo / app).
5. Tipos de datos y zona horaria (Europe/Madrid).
6. Columnas nuevas útiles para el análisis.
7. Nulos, duplicados y valores atípicos (*outliers*).
8. **Validación** contra los totales de la app.
9. Jornadas de la flota (horas trabajadas).
10. Tabla diaria y guardado.

In [1]:
# --- Librerías ---
import pandas as pd                 # pandas: la librería estándar para trabajar con tablas en Python
import numpy as np                  # numpy: cálculo numérico (pandas está construido encima de numpy)
from pathlib import Path            # Path: para escribir rutas de archivos que funcionan en cualquier sistema

pd.set_option("display.max_columns", 30)   # mostrar hasta 30 columnas al imprimir una tabla
pd.set_option("display.width", 160)        # ancho de impresión, para que no se corten las tablas

# --- Rutas ---
BASE = Path.cwd()                          # carpeta donde está este cuaderno (proyecto_vtc/)
INTER = BASE / "datos" / "intermedios"     # datos transcritos (entrada de este cuaderno)
LIMPIOS = BASE / "datos" / "limpios"       # carpeta de salida
LIMPIOS.mkdir(parents=True, exist_ok=True) # crearla si no existe (no da error si ya existe)

ZONA = "Europe/Madrid"                     # zona horaria de las horas que muestra la app

# Lista donde iremos apuntando cada cambio que hagamos a los datos (el «registro de cambios»)
registro = []
def apunta(paso, accion, detalle, motivo, filas=1):
    """Añade una línea al registro de cambios."""
    registro.append(dict(paso=paso, accion=accion, detalle=detalle, motivo=motivo, filas_afectadas=filas))

## 1. Cargar y mirar los datos en bruto

El archivo `viajes_ocr_candidatos.csv` tiene **una fila por cada viaje visto en cada foto**. Como las fotos se solapan (al hacer *scroll* se repiten 1–2 viajes), habrá repetidos.

Lo primero con cualquier tabla nueva: ver su tamaño (`shape`), las primeras filas (`head`), los tipos (`dtypes`) y cuántos huecos hay (`isna().sum()`).

In [2]:
cand = pd.read_csv(INTER / "viajes_ocr_candidatos.csv")   # leer el CSV como tabla (DataFrame)
print("Filas y columnas:", cand.shape)                     # (número de filas, número de columnas)
cand.head(8)                                               # primeras 8 filas

Filas y columnas: (245, 8)


,imagen,y,texto_fecha,fecha,hora,producto,texto_importe,importe_eur
0,IMG_7835,0.292,"Hoy, 10:37",2026-10-07,10:37,VTC,"23,37 € >",23.37
1,IMG_7835,0.370,"Hoy, 09:48",2026-10-07,09:48,VTC Eco,"14,25 €",14.25
2,IMG_7835,0.449,"Hoy, 08:46",2026-10-07,08:46,VTC Airport,"33,85 € >",33.85
3,IMG_7835,0.528,"Hoy, 07:49",2026-10-07,07:49,VTC Eco,"18,46 € >",18.46
4,IMG_7835,0.606,"Hoy, 06:56",2026-10-07,06:56,VTC Airport,"8,73 €",8.73
5,IMG_7835,0.687,"Hoy, 06:43",2026-10-07,06:43,VTC,6 €,6.00
6,IMG_7835,0.767,"Hoy, 06:31",2026-10-07,06:31,VTC Airport,0€,0.00
7,IMG_7835,0.849,"Ayer, 11:02",2026-10-06,11:02,VTC,"6,70 €",6.70


In [3]:
print(cand.dtypes)          # tipo de cada columna: object = texto, float64 = número decimal
print()
print("Huecos (nulos) por columna:")
print(cand.isna().sum())    # isna() marca True donde falta el dato; sum() cuenta los True por columna

imagen               str
y                float64
texto_fecha          str
fecha                str
hora                 str
producto             str
texto_importe        str
importe_eur      float64
dtype: object

Huecos (nulos) por columna:
imagen            0
y                 0
texto_fecha       0
fecha             1
hora              1
producto         14
texto_importe    10
importe_eur      10
dtype: int64


**Qué vemos:** `fecha` y `hora` son texto (*object*), no fechas de verdad: hay que convertirlas. Y hay huecos en `producto` e `importe_eur`: son filas que salen **cortadas** en el borde de la foto, o fotos movidas que el OCR leyó mal.

> **¿Qué es un nulo?** Una celda vacía, sin dato (en pandas se ve como `NaN`). Nunca se rellena «a ojo»: o se recupera de otra fuente fiable o se deja vacía y se explica.

## 2. Aplicar las correcciones manuales

Revisé a mano **las 25 fotos del historial** comparando con lo que leyó el OCR. Cada corrección está en `correcciones_manuales.csv`, con su motivo. Hay tres tipos de acción:
- **eliminar**: filas cortadas en el borde que aparecen completas en la foto siguiente.
- **corregir**: valores mal leídos (sobre todo en la foto movida IMG_7843).
- **añadir**: viajes que el OCR no detectó (por ejemplo, uno tapado por un dedo).

In [4]:
corr = pd.read_csv(INTER / "correcciones_manuales.csv")   # tabla de correcciones hechas a mano
print(corr.accion.value_counts())                          # cuántas de cada tipo
corr[["imagen", "texto_fecha", "accion", "motivo"]].head(10)

accion
eliminar    11
corregir    11
añadir       2
Name: count, dtype: int64


,imagen,texto_fecha,accion,motivo
0,IMG_7836,"Lunes, 12:33",eliminar,Fila cortada en el borde inferior; el viaje ap...
1,IMG_7838,"02 de Octubre, 08:44",eliminar,Fila cortada (tipo de servicio no visible); ap...
2,IMG_7840,30 de Sentiemhre 07•58,eliminar,Fila cortada en el borde; aparece completa en ...
3,IMG_7840,(fila superior cortada),añadir,"Fila parcialmente visible arriba: fecha 01/10,..."
4,IMG_7841,"29 de Septiembre, 09:53",eliminar,Fila cortada; aparece completa en IMG_7842
5,IMG_7842,"28 de Septiembre, 11:54",corregir,Fila cortada abajo; tipo e importe leídos en I...
6,IMG_7843,(fila tapada por el dedo),añadir,Dedo tapa el día y el tipo de servicio; se lee...
7,IMG_7843,"28 de Septiembre, 09:45",corregir,Foto movida: corregido a mano (OCR no leyó el ...
8,IMG_7843,"28 de Septiembre, 09:14",corregir,Foto movida: corregido a mano
9,IMG_7843,"28 de Septiembre, 0B:47",corregir,Foto movida: el OCR leyó 0B:47; en el recorte ...


In [5]:
viajes = cand.copy()                       # trabajamos sobre una COPIA, el original no se toca
viajes["origen_dato"] = "ocr"              # de dónde viene cada valor (para la trazabilidad)

for _, c in corr.iterrows():                                   # recorrer cada corrección, una a una
    fila = (viajes.imagen == c.imagen) & (viajes.texto_fecha == c.texto_fecha)  # localizar la fila
    if c.accion == "eliminar":
        assert fila.sum() == 1                                 # comprobación: debe existir exactamente 1
        viajes = viajes[~fila]                                 # ~ significa «NO»: nos quedamos con el resto
    elif c.accion == "corregir":
        assert fila.sum() == 1
        antes = viajes.loc[fila, ["fecha", "hora", "producto", "importe_eur"]].iloc[0].to_dict()
        for col in ["fecha", "hora", "producto", "importe_eur"]:
            viajes.loc[fila, col] = c[col]                     # sobrescribir con el valor revisado a mano
        viajes.loc[fila, "origen_dato"] = "ocr + corrección manual"
        apunta("2-correcciones", "corregir", f"{c.imagen} '{c.texto_fecha}': {antes} -> "
               f"{c.fecha} {c.hora} {c.producto} {c.importe_eur} €", c.motivo)
        continue
    elif c.accion == "añadir":
        nueva = dict(imagen=c.imagen, texto_fecha=c.texto_fecha, fecha=c.fecha, hora=c.hora,
                     producto=c.producto, importe_eur=c.importe_eur, origen_dato="transcripción manual")
        viajes = pd.concat([viajes, pd.DataFrame([nueva])], ignore_index=True)  # pegar la fila nueva al final
    apunta("2-correcciones", c.accion, f"{c.imagen} '{c.texto_fecha}'", c.motivo)

print("Filas tras las correcciones:", len(viajes))
print("¿Quedan huecos?", viajes[["fecha", "hora", "producto", "importe_eur"]].isna().sum().sum())

Filas tras las correcciones: 236
¿Quedan huecos? 0


## 3. Quitar duplicados

> **¿Qué es un duplicado?** La misma observación (aquí, el mismo viaje) registrada más de una vez. Si no se quitan, se cuentan dos veces los ingresos.

**Cuidado:** dos viajes reales pueden ser idénticos. El 14/09 a las 09:13 hay **dos** viajes VTC Eco de 0 € que salen juntos en la misma foto: son dos viajes distintos, no un repetido.

**Regla usada:** un viaje se identifica por (fecha, hora, tipo, importe). Para cada combinación miro cuántas veces aparece **como máximo dentro de una misma foto**, y conservo ese número. Así quito los repetidos entre fotos sin borrar los gemelos reales.

In [6]:
clave = ["fecha", "hora", "producto", "importe_eur"]          # columnas que identifican un viaje

# Para cada viaje: ¿cuántas veces sale en cada foto? -> nos quedamos con el máximo
veces = (viajes.groupby(clave + ["imagen"]).size()            # contar apariciones por viaje y foto
               .groupby(level=clave).max()                     # máximo entre fotos
               .rename("n_real").reset_index())
# Unir en una sola cadena las fotos donde sale cada viaje (trazabilidad)
fotos = (viajes.groupby(clave)
               .agg(imagenes_origen=("imagen", lambda s: ";".join(sorted(set(s)))),
                    origen_dato=("origen_dato", lambda s: " / ".join(sorted(set(s)))))
               .reset_index())
unicos = veces.merge(fotos, on=clave)                          # juntar ambas tablas por la clave
unicos = unicos.loc[unicos.index.repeat(unicos.n_real)].drop(columns="n_real")  # repetir los gemelos reales

quitados = len(viajes) - len(unicos)
apunta("3-duplicados", "eliminar", "Viajes repetidos por el solape entre fotos consecutivas",
       "Mismo viaje (fecha, hora, tipo, importe) visto en 2 fotos", filas=quitados)
print(f"{len(viajes)} filas -> {len(unicos)} viajes únicos ({quitados} repetidos eliminados)")
print("Gemelos reales conservados:", unicos.duplicated(clave, keep=False).sum(), "filas")
viajes = unicos.reset_index(drop=True)

236 filas -> 212 viajes únicos (24 repetidos eliminados)
Gemelos reales conservados: 2 filas


## 4. Método de pago

Cada fila del historial tiene un icono: **móvil** = pagado en la app, **monedas** = pagado en efectivo. El OCR no lee iconos, así que los anoté a mano en `pagos_efectivo_manual.csv` (20 viajes en efectivo). El resto son pagos en la app.

In [7]:
ef = pd.read_csv(INTER / "pagos_efectivo_manual.csv")         # viajes con icono de efectivo
ef["metodo_pago"] = "efectivo"
viajes = viajes.merge(ef[["fecha", "hora", "importe_eur", "metodo_pago"]],   # merge = unir tablas por columnas comunes
                      on=["fecha", "hora", "importe_eur"], how="left")       # how="left": conservar todos los viajes
assert (viajes.metodo_pago == "efectivo").sum() == len(ef)   # los 20 deben haber encontrado su viaje
viajes["metodo_pago"] = viajes.metodo_pago.fillna("app")     # fillna: rellenar los huecos -> «app»
apunta("4-pago", "añadir columna", "metodo_pago (app/efectivo) según el icono de cada fila",
       "Anotado a mano: el OCR no lee iconos", filas=len(viajes))
viajes.metodo_pago.value_counts()

metodo_pago
app         192
efectivo     20
Name: count, dtype: int64

## 5. Tipos de datos y zona horaria

Ahora `fecha` y `hora` son texto. Las juntamos y las convertimos a **fecha-hora** (*datetime*), y le decimos que es hora de Madrid.

> **¿Por qué importa la zona horaria?** Una hora sin zona es ambigua: las 08:00 de Madrid son las 06:00 en UTC (hora universal) en verano. Si un día mezclas datos de varias fuentes (por ejemplo una exportación de Uber en UTC), sin zona horaria cruzarías mal los viajes. Aquí todas las horas son de la app en el móvil, en hora de Madrid. El periodo (31/08–07/10) no cruza ningún cambio de hora (el siguiente es el 25/10/2026), así que todas están en horario de verano (UTC+2).

Moneda y decimales: el OCR leía «23,37 €» (coma decimal española). En la extracción ya se convirtió a número con punto (23.37), que es lo que entiende Python. Todo está en euros.

In [8]:
viajes["fecha_hora"] = (pd.to_datetime(viajes.fecha + " " + viajes.hora)   # texto -> fecha-hora
                          .dt.tz_localize(ZONA))                             # asignar zona Europe/Madrid
viajes["importe_eur"] = viajes.importe_eur.astype(float).round(2)           # número decimal con 2 decimales
viajes = viajes.sort_values("fecha_hora").reset_index(drop=True)            # ordenar del más antiguo al más nuevo
viajes.insert(0, "id_viaje", [f"V{i:03d}" for i in range(1, len(viajes) + 1)])  # identificador V001, V002...
apunta("5-tipos", "convertir", "fecha+hora -> datetime con zona Europe/Madrid; importe -> float",
       "Para poder ordenar, agrupar por hora/día y calcular", filas=len(viajes))
print(viajes.dtypes[["fecha_hora", "importe_eur"]])
viajes[["id_viaje", "fecha_hora", "producto", "importe_eur", "metodo_pago"]].head()

fecha_hora     datetime64[us, Europe/Madrid]
importe_eur                          float64
dtype: object


,id_viaje,fecha_hora,producto,importe_eur,metodo_pago
0,V001,2026-08-31 21:23:00+02:00,VTC,8.89,app
1,V002,2026-08-31 21:39:00+02:00,VTC,8.15,app
2,V003,2026-08-31 21:59:00+02:00,VTC Prioritario,16.60,app
3,V004,2026-08-31 22:17:00+02:00,VTC,0.00,app
4,V005,2026-08-31 22:19:00+02:00,VTC,15.85,app


## 6. Columnas nuevas (variables derivadas)

Para responder «¿qué día y franja rinde más?» necesitamos columnas como el día de la semana o la hora. Se **derivan** (calculan) a partir de la fecha-hora.

- `sin_ingreso`: viaje a 0 € (casi seguro, cancelado o no realizado; la app no lo explica).
- `importe_6_exactos`: viajes de 6 € justos, muy frecuentes. Parece un **importe mínimo** (tarifa mínima o compensación por cancelación: no se puede distinguir con estos datos).
- `dia_completo`: el 31/08 solo tiene los viajes desde las 21:23 (la última foto acaba ahí), así que ese día está **incompleto**.
- `hora_aproximada`: el viaje del 01/10 con la hora cortada.
- `min_hasta_siguiente`: minutos hasta el siguiente viaje del mismo día (para estudiar esperas).

In [9]:
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
fh = viajes.fecha_hora                                      # atajo para escribir menos

viajes["fecha"] = fh.dt.strftime("%Y-%m-%d")                # fecha como texto AAAA-MM-DD
viajes["mes"] = fh.dt.strftime("%Y-%m")                     # mes (para agrupar)
viajes["semana_iso"] = fh.dt.isocalendar().week.astype(int) # nº de semana del año (lunes a domingo)
viajes["num_dia_semana"] = fh.dt.dayofweek                  # 0 = lunes ... 6 = domingo
viajes["dia_semana"] = viajes.num_dia_semana.map(dict(enumerate(DIAS)))  # número -> nombre del día
viajes["hora_del_dia"] = fh.dt.hour                         # hora en punto (6, 7, 8...)
viajes["franja_horaria"] = viajes.hora_del_dia.map(lambda h: f"{h:02d}:00-{h:02d}:59")
viajes["sin_ingreso"] = viajes.importe_eur.eq(0)            # True si el importe es 0
viajes["importe_6_exactos"] = viajes.importe_eur.eq(6)      # True si es exactamente 6 €
viajes["es_aeropuerto"] = viajes.producto.eq("VTC Airport")
viajes["dia_completo"] = viajes.fecha.ne("2026-08-31")      # ne = «no igual»: todos menos el 31/08
viajes["hora_aproximada"] = viajes.origen_dato.str.contains("manual") & viajes.hora.eq("08:00") & viajes.fecha.eq("2026-10-01")

# Minutos hasta el siguiente viaje DEL MISMO DÍA (shift(-1) = valor de la fila siguiente)
sig = viajes.groupby("fecha").fecha_hora.shift(-1)
viajes["min_hasta_siguiente"] = (sig - viajes.fecha_hora).dt.total_seconds() / 60

apunta("6-derivadas", "añadir columnas", "mes, semana_iso, dia_semana, hora_del_dia, franja_horaria, "
       "sin_ingreso, importe_6_exactos, es_aeropuerto, dia_completo, hora_aproximada, min_hasta_siguiente",
       "Necesarias para agrupar por día/franja y estudiar esperas", filas=len(viajes))
apunta("6-derivadas", "marcar", "31/08: dia_completo = False", "Solo se capturaron viajes desde las 21:23",
       filas=int((~viajes.dia_completo).sum()))
viajes[["id_viaje", "fecha", "dia_semana", "franja_horaria", "producto", "importe_eur",
        "sin_ingreso", "min_hasta_siguiente"]].head(10)

,id_viaje,fecha,dia_semana,franja_horaria,producto,importe_eur,sin_ingreso,min_hasta_siguiente
0,V001,2026-08-31,lunes,21:00-21:59,VTC,8.89,False,16.0
1,V002,2026-08-31,lunes,21:00-21:59,VTC,8.15,False,20.0
2,V003,2026-08-31,lunes,21:00-21:59,VTC Prioritario,16.60,False,18.0
3,V004,2026-08-31,lunes,22:00-22:59,VTC,0.00,True,2.0
4,V005,2026-08-31,lunes,22:00-22:59,VTC,15.85,False,26.0
5,V006,2026-08-31,lunes,22:00-22:59,VTC Prioritario,10.71,False,28.0
6,V007,2026-08-31,lunes,23:00-23:59,VTC Prioritario,8.04,False,13.0
7,V008,2026-08-31,lunes,23:00-23:59,VTC Airport,16.03,False,NaN
8,V009,2026-09-01,martes,08:00-08:59,Electric,12.73,False,33.0
9,V010,2026-09-01,martes,08:00-08:59,VTC Prioritario,16.91,False,35.0


## 7. Nulos, duplicados y valores atípicos

> **¿Qué es un valor atípico (*outlier*)?** Un dato muy alejado del resto (un viaje de 45 € cuando lo normal son 10–15 €). Puede ser un error… o un dato real e importante (un viaje largo al aeropuerto). **No se borra automáticamente**: se marca y se decide con criterio.

**Método del rango intercuartílico (IQR):** ordenamos los importes; Q1 es el valor que deja por debajo al 25 % y Q3 al 75 %. IQR = Q3 − Q1 (la anchura del 50 % central). Se considera atípico lo que queda por debajo de Q1 − 1,5·IQR o por encima de Q3 + 1,5·IQR. Lo calculo **solo con los viajes con ingreso** (> 0 €), porque los 0 € son otra cosa (cancelaciones).

In [10]:
print("Nulos en columnas clave:", viajes[["fecha_hora", "producto", "importe_eur", "metodo_pago"]].isna().sum().sum())
print("Filas idénticas en (fecha_hora, producto, importe):",
      viajes.duplicated(["fecha_hora", "producto", "importe_eur"]).sum(), "-> son los 2 gemelos reales del 14/09 09:13")

con_ingreso = viajes.loc[~viajes.sin_ingreso, "importe_eur"]   # solo viajes > 0 €
q1, q3 = con_ingreso.quantile([0.25, 0.75])                    # cuartiles 1 y 3
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1 = {q1:.2f} €, Q3 = {q3:.2f} €, IQR = {iqr:.2f} € -> atípico si < {lim_inf:.2f} € o > {lim_sup:.2f} €")

viajes["atipico_iqr"] = (~viajes.sin_ingreso) & ((viajes.importe_eur < lim_inf) | (viajes.importe_eur > lim_sup))
apunta("7-atipicos", "marcar (no borrar)", f"atipico_iqr = importe fuera de [{lim_inf:.2f}, {lim_sup:.2f}] €",
       "Viajes reales verificados en las fotos: se conservan", filas=int(viajes.atipico_iqr.sum()))
viajes.loc[viajes.atipico_iqr | (viajes.importe_eur.between(0.01, 5.99)),
           ["id_viaje", "fecha_hora", "producto", "importe_eur", "imagenes_origen"]]

Nulos en columnas clave: 0
Filas idénticas en (fecha_hora, producto, importe): 1 -> son los 2 gemelos reales del 14/09 09:13
Q1 = 7.46 €, Q3 = 15.90 €, IQR = 8.44 € -> atípico si < -5.20 € o > 28.56 €


,id_viaje,fecha_hora,producto,importe_eur,imagenes_origen
56,V057,2026-09-09 10:13:00+02:00,VTC,28.95,IMG_7853
68,V069,2026-09-11 08:15:00+02:00,VTC,32.83,IMG_7852
86,V087,2026-09-15 08:09:00+02:00,VTC,3.35,IMG_7850
135,V136,2026-09-23 09:23:00+02:00,VTC,38.95,IMG_7844
148,V149,2026-09-28 08:17:00+02:00,VTC,33.13,IMG_7843
164,V165,2026-09-30 08:30:00+02:00,VTC Airport,45.12,IMG_7840;IMG_7841
209,V210,2026-10-07 08:46:00+02:00,VTC Airport,33.85,IMG_7835


**Decisión:** los importes altos son viajes reales (los comprobé en las fotos; varios son de aeropuerto) y se **conservan**. El de 3,35 € (15/09) está por debajo del mínimo habitual de 6 €; también es real según la foto, pero es raro: quizá un ajuste o un viaje parcial. Se conserva y se menciona en las limitaciones.

## 8. Validación contra los totales de la app

Esta es la prueba más importante: si la transcripción es buena, la suma de los viajes tiene que coincidir con lo que dice la app.

In [11]:
app = pd.read_csv(INTER / "resumen_app_transcrito.csv")      # totales oficiales transcritos de la app
oficial = app[app.concepto.isin(["viajes_pago_app", "viajes_efectivo"])].copy()
oficial["metodo_pago"] = oficial.concepto.map({"viajes_pago_app": "app", "viajes_efectivo": "efectivo"})

nuestro = viajes.groupby(["mes", "metodo_pago"]).importe_eur.sum().round(2).rename("suma_viajes").reset_index()
val = oficial.merge(nuestro, left_on=["periodo", "metodo_pago"], right_on=["mes", "metodo_pago"])
val["diferencia"] = (val.suma_viajes - val.valor).round(2)
print(val[["periodo", "metodo_pago", "valor", "suma_viajes", "diferencia"]].to_string(index=False))
assert (val.diferencia == 0).all(), "¡La transcripción no cuadra con la app!"
print("\n✔ Los 4 totales cuadran al céntimo.")

periodo metodo_pago   valor  suma_viajes  diferencia
2026-09         app 1620.42      1620.42         0.0
2026-09    efectivo  122.61       122.61         0.0
2026-10         app  394.87       394.87         0.0
2026-10    efectivo   57.79        57.79         0.0

✔ Los 4 totales cuadran al céntimo.


También se validan los días con «ganancia diaria» oficial. La app calcula **ganancia = suma de viajes − peajes + incentivos** (los peajes son un reembolso de un gasto, no una ganancia; los incentivos son bonos que no van ligados a un viaje). En octubre los peajes fueron 6 € y los incentivos 3 € (el domingo 4/10, sin viajes).

In [12]:
diario_app = app[app.concepto == "ganancia_diaria"][["periodo", "valor"]].rename(columns={"periodo": "fecha", "valor": "ganancia_app"})
diario = viajes.groupby("fecha").importe_eur.sum().round(2).rename("suma_viajes").reset_index()
cmp_ = diario_app.merge(diario, on="fecha", how="left").fillna({"suma_viajes": 0})
cmp_["diferencia"] = (cmp_.suma_viajes - cmp_.ganancia_app).round(2)
print(cmp_.to_string(index=False))
print("\nSuma de diferencias en octubre (1–7):", cmp_[cmp_.fecha >= "2026-10-01"].diferencia.sum().round(2),
      "= peajes (6 €) − incentivos (3 €)")

     fecha  ganancia_app  suma_viajes  diferencia
2026-09-30        115.13       107.13        -8.0
2026-10-01         87.33        88.33         1.0
2026-10-02         74.17        76.17         2.0
2026-10-04          3.00         0.00        -3.0
2026-10-05        111.84       113.84         2.0
2026-10-06         69.66        69.66         0.0
2026-10-07        103.66       104.66         1.0

Suma de diferencias en octubre (1–7): 3.0 = peajes (6 €) − incentivos (3 €)


**Interpretación:** las diferencias de +1 y +2 € en octubre son los peajes (suman 6 €, igual que el «Peajes 6 €» de la app). El domingo −3 € es el incentivo. El 30/09 −8 € coincide con los 8 € de incentivos de septiembre. Todo cuadra: **la transcripción es fiable.**

> Por eso en este proyecto «ingreso de un viaje» = el importe que muestra el historial (incluye propinas y peajes). Los peajes son solo 11 € en 5 semanas (0,5 %), así que no cambian ninguna conclusión.

## 9. Jornadas de la flota (horas trabajadas)

La app de la flota da el detalle de 8 jornadas. Descubrí (y comprobé en los 8 días) que:

**Tiempo total = (hora fin − hora inicio) + recarga eléctrica + cuidado del vehículo**

Columnas que calculo:
- `duracion_sesion_min`: fin − inicio.
- `tiempo_efectivo_min`: conducción efectiva + tareas + recarga + cuidado (lo que la app llama «tiempo efectivo»).
- `sin_clasificar_min`: minutos de la sesión que la app no asigna a nada (sesión − conducción efectiva − no efectiva − descansos). Probablemente **esperas con el coche parado** (la app no lo dice).
- `jornada_anomala`: el 02/10 la sesión dura 11 h 36 min pero el último viaje empezó a las 11:33 y la conducción efectiva (5 h 2 min) es como la de cualquier otro día. **Lo más probable es que la jornada se quedara abierta.** Se marca y se excluye del cálculo de €/hora principal.

In [13]:
jor = pd.read_csv(INTER / "jornadas_flota_transcritas.csv")
ini = pd.to_datetime(jor.fecha + " " + jor.inicio)          # hora de inicio como fecha-hora
fin = pd.to_datetime(jor.fecha + " " + jor.fin)             # hora de fin
jor["duracion_sesion_min"] = ((fin - ini).dt.total_seconds() / 60).astype(int)
jor["tiempo_efectivo_min"] = jor[["conduccion_efectiva_min", "tareas_adicionales_min",
                                  "recarga_electrica_min", "cuidado_vehiculo_min"]].sum(axis=1)  # axis=1: sumar en horizontal
jor["sin_clasificar_min"] = (jor.duracion_sesion_min - jor.conduccion_efectiva_min
                             - jor.conduccion_no_efectiva_min - jor.descansos_min)
jor["total_recalculado_min"] = jor.duracion_sesion_min + jor.recarga_electrica_min + jor.cuidado_vehiculo_min
jor["dif_vs_app_min"] = jor.total_recalculado_min - jor.tiempo_total_app_min
assert jor.dif_vs_app_min.abs().max() <= 1                  # como mucho 1 min de redondeo

# Hora de inicio del último viaje de cada día, para detectar jornadas que se quedaron abiertas
ultimo = viajes.groupby("fecha").fecha_hora.max().dt.strftime("%H:%M").rename("ultimo_viaje")
jor = jor.merge(ultimo, left_on="fecha", right_index=True, how="left")
mediana = jor.duracion_sesion_min.median()
jor["jornada_anomala"] = jor.duracion_sesion_min > 1.5 * mediana   # regla simple: 50 % más larga que lo típico
jor["dia_semana"] = pd.to_datetime(jor.fecha).dt.dayofweek.map(dict(enumerate(DIAS)))
apunta("9-jornadas", "marcar", "02/10: jornada_anomala = True (sesión 6:30–18:06, último viaje 11:33)",
       "Probable jornada no cerrada; se excluye del €/hora principal", filas=int(jor.jornada_anomala.sum()))
jor[["fecha", "inicio", "fin", "ultimo_viaje", "duracion_sesion_min", "conduccion_efectiva_min",
     "recarga_electrica_min", "sin_clasificar_min", "tiempo_total_app_min", "dif_vs_app_min", "jornada_anomala"]]

,fecha,inicio,fin,ultimo_viaje,duracion_sesion_min,conduccion_efectiva_min,recarga_electrica_min,sin_clasificar_min,tiempo_total_app_min,dif_vs_app_min,jornada_anomala
0,2026-09-28,06:15,12:26,11:54,371,314,43,46,424,0,False
1,2026-09-29,06:11,12:17,11:22,366,320,33,42,409,0,False
2,2026-09-30,06:17,12:45,11:20,388,351,25,31,422,1,False
3,2026-10-01,06:23,12:22,11:01,359,313,10,43,378,1,False
4,2026-10-02,06:30,18:06,11:33,696,302,37,389,742,1,True
5,2026-10-05,06:20,13:02,12:43,402,312,36,65,447,1,False
6,2026-10-06,06:22,13:04,11:02,402,295,55,107,466,1,False
7,2026-10-07,06:21,12:28,10:37,367,306,45,61,422,0,False


## 10. Tabla diaria y guardado

Juntamos por día: ingresos, nº de viajes, viajes a 0 €, primer y último viaje y, cuando existe, las horas de la flota.

In [14]:
dia = (viajes.groupby("fecha")
             .agg(n_viajes=("id_viaje", "count"),                    # cuántos viajes
                  n_sin_ingreso=("sin_ingreso", "sum"),               # cuántos a 0 €
                  ingresos_eur=("importe_eur", "sum"),                # suma de importes
                  primer_viaje=("fecha_hora", "min"),
                  ultimo_viaje=("fecha_hora", "max"),
                  dia_completo=("dia_completo", "all"))
             .reset_index())
dia["ingresos_eur"] = dia.ingresos_eur.round(2)
dia["dia_semana"] = pd.to_datetime(dia.fecha).dt.dayofweek.map(dict(enumerate(DIAS)))
dia["semana_iso"] = pd.to_datetime(dia.fecha).dt.isocalendar().week.astype(int)
dia["ventana_actividad_h"] = ((dia.ultimo_viaje - dia.primer_viaje).dt.total_seconds() / 3600).round(2)
dia["primer_viaje"] = dia.primer_viaje.dt.strftime("%H:%M")
dia["ultimo_viaje"] = dia.ultimo_viaje.dt.strftime("%H:%M")
dia = dia.merge(jor[["fecha", "tiempo_total_app_min", "tiempo_efectivo_min", "conduccion_efectiva_min",
                     "jornada_anomala"]], on="fecha", how="left")
dia["tiene_horas_flota"] = dia.tiempo_total_app_min.notna()
dia.tail(10)

,fecha,n_viajes,n_sin_ingreso,ingresos_eur,primer_viaje,ultimo_viaje,dia_completo,dia_semana,semana_iso,ventana_actividad_h,tiempo_total_app_min,tiempo_efectivo_min,conduccion_efectiva_min,jornada_anomala,tiene_horas_flota
17,2026-09-23,7,0,97.87,07:40,12:02,True,miércoles,39,4.37,NaN,NaN,NaN,NaN,False
18,2026-09-24,7,1,76.24,07:22,11:02,True,jueves,39,3.67,NaN,NaN,NaN,NaN,False
19,2026-09-28,7,0,90.76,06:29,11:54,True,lunes,40,5.42,424.0,367.0,314.0,False,True
20,2026-09-29,8,1,72.96,06:19,11:22,True,martes,40,5.05,409.0,363.0,320.0,False,True
21,2026-09-30,8,1,107.13,06:49,11:20,True,miércoles,40,4.52,422.0,386.0,351.0,False,True
22,2026-10-01,11,2,88.33,06:49,11:01,True,jueves,40,4.20,378.0,333.0,313.0,False,True
23,2026-10-02,7,2,76.17,07:31,11:33,True,viernes,40,4.03,742.0,349.0,302.0,True,True
24,2026-10-05,10,3,113.84,06:20,12:43,True,lunes,41,6.38,447.0,358.0,312.0,False,True
25,2026-10-06,8,1,69.66,06:27,11:02,True,martes,41,4.58,466.0,360.0,295.0,False,True
26,2026-10-07,7,1,104.66,06:31,10:37,True,miércoles,41,4.10,422.0,361.0,306.0,False,True


In [15]:
# --- Guardar las tablas limpias (CSV con punto decimal, UTF-8: lo abre Excel, Power BI, SQL...) ---
cols_viajes = ["id_viaje", "fecha_hora", "fecha", "hora", "mes", "semana_iso", "num_dia_semana", "dia_semana",
               "hora_del_dia", "franja_horaria", "producto", "importe_eur", "metodo_pago", "sin_ingreso",
               "importe_6_exactos", "es_aeropuerto", "dia_completo", "hora_aproximada", "atipico_iqr",
               "min_hasta_siguiente", "origen_dato", "imagenes_origen"]
salida = viajes[cols_viajes].copy()
salida["fecha_hora"] = salida.fecha_hora.dt.strftime("%Y-%m-%dT%H:%M:%S%z")  # ISO 8601 con zona (+0200)
salida["min_hasta_siguiente"] = salida.min_hasta_siguiente.round(0)
salida.to_csv(LIMPIOS / "viajes_limpios.csv", index=False)
jor.to_csv(LIMPIOS / "jornadas_limpias.csv", index=False)
dia.to_csv(LIMPIOS / "resumen_diario.csv", index=False)
app.to_csv(LIMPIOS / "resumen_app.csv", index=False)

apunta("10-guardar", "exportar", "viajes_limpios, jornadas_limpias, resumen_diario, resumen_app", "Salida del cuaderno",
       filas=len(salida))
reg = pd.DataFrame(registro)
reg.to_csv(LIMPIOS / "registro_cambios.csv", index=False)
print(f"viajes_limpios.csv: {len(salida)} filas | jornadas_limpias.csv: {len(jor)} | "
      f"resumen_diario.csv: {len(dia)} | registro_cambios.csv: {len(reg)} cambios")
reg.groupby("paso").filas_afectadas.agg(["count", "sum"])

viajes_limpios.csv: 212 filas | jornadas_limpias.csv: 8 | resumen_diario.csv: 27 | registro_cambios.csv: 32 cambios


,count,sum
paso,,
10-guardar,1,212
2-correcciones,24,24
3-duplicados,1,24
4-pago,1,212
5-tipos,1,212
6-derivadas,2,220
7-atipicos,1,6
9-jornadas,1,1


## Resumen de la limpieza

- **245** filas leídas por OCR → **212 viajes únicos** (11 filas cortadas eliminadas, 11 corregidas, 2 añadidas a mano, 24 repetidos quitados).
- Los totales de la app cuadran **al céntimo** en los 4 controles (septiembre y octubre, app y efectivo).
- 31/08 marcado como día incompleto; 02/10 marcado como jornada anómala (probablemente no cerrada).
- Los valores atípicos se **marcan, no se borran**.
- Siguiente paso: `02_analisis.ipynb`.